In [14]:
import sqlite3
import pandas as pd
import folium 

import pandas as pd
import sqlite3

listings = pd.read_csv("Data/updated_listing.csv", encoding="latin1") #use the pre-processed and cleaned dataset
reviews = pd.read_csv("Data/updated_review.csv",  encoding="latin1")

listings["price_num"] = ( #convert the price columns to numeric columns by stripping "$" and commas.
    listings["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

db = sqlite3.connect("airbnb.sqlite") #create the sql database
cur = db.cursor()
listings.to_sql("listings", db, if_exists="replace", index=False)
reviews.to_sql("reviews", db, if_exists="replace", index=False)

db.commit()

#Query 1: Top Five Cities
cur.executescript("""
DROP TABLE IF EXISTS top_five_cities; --make sure to delete the table if the code is rerun
CREATE TABLE top_five_cities AS
SELECT city, COUNT(*) AS number_listings --count the number of listings per city
FROM listings
WHERE city IS NOT NULL --make sure to drop null values
GROUP BY city --group by city
ORDER BY number_listings DESC --get the most number of listings at the top
LIMIT 5; --get the top five
""")
db.commit()


#Query 2: Average Price for Neighborhoods
cur.executescript("""
DROP TABLE IF EXISTS neighbourhood_avg_prices; --make sure to delete the table if the code is rerun
CREATE TABLE neighbourhood_avg_prices AS
SELECT
    city,
    neighbourhood,
    AVG(price_num) AS avg_price, --calculate the average price
    COUNT(*) AS n_listings, --count the number of listings within each neighborhood
    AVG(latitude)  AS lat, --mean latitude and longitude to use for visualization
    AVG(longitude) AS lon
FROM listings
WHERE city IN (SELECT city FROM top_five_cities)
    AND neighbourhood IS NOT NULL --make sure to drop NULL values
    AND price_num IS NOT NULL
    AND price_num BETWEEN 10 AND 1000 --add a range for prices
    AND latitude IS NOT NULL
    AND longitude IS NOT NULL
GROUP BY city, neighbourhood --group by the city and enighborhood pair
HAVING COUNT(*) >= 30 --set a minimum listing value to reduce noise
"""
)
db.commit()

#Query 3: Ranking
query_3 = """
SELECT
    city,
    neighbourhood,
    ROUND(avg_price, 2) AS avg_price, --round each price to two decimal places
    n_listings,
    lat,
    lon
    --REFINE CODE START: USED AI TO REFINE CODE HERE USING WINDOW FUNCTION
FROM (
    SELECT
        city,
        neighbourhood,
        avg_price,
        n_listings,
        lat,
        lon,
        ROW_NUMBER() OVER ( --window function which assigns a ranking to each neighborhood within each city based on abg price
            PARTITION BY city --does this for each city
            ORDER BY avg_price DESC --get the highst average priced neighborhood ranked as 1
        ) AS rnk
    FROM neighbourhood_avg_prices
) ranked
--REFINE CODE END
WHERE rnk = 1 --only select the enighborhood within each city ranked as 1
ORDER BY n_listings DESC;
"""
output = pd.read_sql(query_3, db) #finally print our output
output




,city,neighbourhood,avg_price,n_listings,lat,lon
0,Rome,I Centro Storico,96.61,11209,41.898751,12.479873
1,Paris,Elysee,161.55,1070,48.874044,2.312994
2,Rio de Janeiro,Leblon,379.73,680,-22.983380,-43.223013
3,Sydney,Pittwater,330.82,680,-33.646396,151.313994
4,Istanbul,Adalar,429.58,102,40.873515,29.113922


In [15]:
#Using Folium to create an interactive map

m = folium.Map(location=[20, 0], zoom_start=2, tiles="cartodbpositron")

for _, row in output.iterrows():
    popup_text = f"""
    <b>City:</b> {row['city']}<br>
    <b>Neighborhood:</b> {row['neighbourhood']}<br>
    <b>Avg Price (national currency):</b> {row['avg_price']}<br>
    <b>Listings:</b> {row['n_listings']}
    """
    folium.Marker(
        location=[float(row["lat"]), float(row["lon"])],
        popup=folium.Popup(popup_text, max_width=300),
        icon=folium.Icon(color="blue", icon="home")
    ).add_to(m)

m.save("maps/airbnb_neighborhoods_map.html")
db.close()